# Acetic acid and its dimer

## Imports

In [1]:
import matplotlib.pyplot as plt
%matplotlib inline

import numpy as np
np.set_printoptions(precision=4)
import os
import pandas as pd

In [2]:
from wilson import spectrum
from wilson import rendering
from wilson import analysis
from wilson.relay import DataVault
from wilson.utils import get_package_root

## Settings

In [3]:
log10=True
w1mw2=False
broad_factor_rc=10.

# terms_selection = [0, 1], [0, 1, 2, 3, 4, 5]
terms_selection = [0, 1], [0, 1]

regions = {1: ((1280., 3150., 10.), (1589., 6050., 10.)),
           2: ((2826.813, 2960., 10.), (5210., 5350., 10.)),
           3: ((2826.813, 3060., 10.), (3725., 4925., 10.)),
           4: ((680., 1750., 10.), (1509., 4350., 10.)),
           5: ((680., 1750., 10.), (1509., 3050., 10.)), 
           6: ((500., 3150., 10.), (500., 6050., 10.))}

# template
settings_here = {'electrical': None, 'mechanical': None,
                 'Gamma_rc': broad_factor_rc, 'region': 1,
                 'font_dict': {'size': 18}, 'figsize': (12, 15), 'norm_max': 1e12, 'norm_min': 1e7,
                 'dynamic_range_n': 10000}

spectra_dict = {"software": [], "code": [], "method": [], "basis_set": [],
                "el+mech max": [], "el max": [], "mech max": []}

other = {'regions': regions, 'terms_selection': terms_selection, 'w1mw2': False, 'log10': True}

## Getting data from data vault

In [4]:
wilson_root = get_package_root()
wilson_root

'/home/vlew/Wilson'

In [5]:
wilson_root = get_package_root()
data_vault = DataVault(wilson_root+'/tests/test_database/mini_files_database.csv')

dataframe_gaussian = data_vault.getting_files_DB("gaussian")
method_basis = dataframe_gaussian[(dataframe_gaussian['code'] == 'ACAC') & (dataframe_gaussian['method'] != 'PBE0')][["code", "method", "basis_set"]]
tuples_method_basis = [(row['code'], row['method'], row['basis_set']) for index, row in method_basis.iterrows()]
tuples_method_basis

[('ACAC', 'B3LYP', 'cc_pVQZ')]

## Acetic acid dimer (HF/cc-pVQZ)

In [ ]:
# selection of the spectrum
datain = data_vault.make_DatainputDict('gaussian', ('ACDM', 'B3LYP', 'cc_pVQZ'), wilson_root)
# list_figs = [(True, False), (False, True), (True, True)]
list_figs = [(True, True)]
region = 1
omega1 = np.arange(*regions[region][0])
omega2 = np.arange(*regions[region][1])
# omega1 = [1185.288, 1247.878, 1501.586]
# omega2 = [2687.219, 2491.481, 2364.564]

for s in list_figs:
    settings_here['electrical'] = s[0]
    settings_here['mechanical'] = s[1]
    vibEL=False
    directory='./'
    
    region = settings_here['region']
    Gamma_rc = settings_here['Gamma_rc']
    el_bool = settings_here['electrical']
    mech_bool = settings_here['mechanical']

    computedSpectrum = spectrum.SpectrumEVV(omega1, omega2, input_data_info=datain, vib_levels_harmonic=vibEL)
    computedSpectrum.addTerms(*terms_selection)
    print(computedSpectrum.fundamentals)
    print(sorted(list(computedSpectrum.fundamentals.values())))
    # print(computedSpectrum.tensor_3d.T)
    Gamma = spectrum.rec_cm2rec_s(Gamma_rc)
    # sec_hypol_data, savedict = computedSpectrum.intensity(Gamma, {}, el=el_bool, mech=mech_bool)

    sec_hypol_data = 0
    if settings_here['electrical']:
        electrical, Qab_contrib_dict = computedSpectrum.intensity_electrical(Gamma)
        sec_hypol_data += electrical

    if settings_here['mechanical']:
        mechall, Qabc_contrib_dict = computedSpectrum.intensity_mechanical(Gamma)
        template_array = np.full(mechall.shape, -0.+0.j, dtype=complex)
        sec_hypol_data += mechall

    name = rendering.make_name(datain, vibEL, settings_here, other, directory)
    artist = rendering.SpectrumFigure(sec_hypol_data, computedSpectrum.w1_mesh, computedSpectrum.w2_mesh, settings_here)
    title_on_top, text_under_the_figure = rendering.make_texts4fig(datain, computedSpectrum, artist, settings_here, other, directory)
    fig = artist.plot2Dmatplotlib(nametuple=(name, os.path.join(os.path.dirname('__file__')), title_on_top),
                                    text_under_the_figure=text_under_the_figure, diagonal=False, to_save=False)
    plt.show()


Used vibrational energy levels:
 harmonic? - False
{'0': 2885.91, '1': 3185.566, '2': 3002.789, '3': 3282.474, '4': 2976.141, '5': 2791.954, '6': 3189.203, '7': 2906.359, '8': 1708.984, '9': 1649.419, '10': 1503.958, '11': 1477.665, '12': 1441.353, '13': 1506.717, '14': 1422.44, '15': 1420.881, '16': 1377.071, '17': 1360.974, '18': 1282.729, '19': 1276.773, '20': 1083.409, '21': 1045.114, '22': 1050.298, '23': 1008.669, '24': 955.851, '25': 915.571, '26': 888.182, '27': 884.448, '28': 624.688, '29': 612.197, '30': 627.269, '31': 582.801, '32': 494.362, '33': 433.742, '34': 160.048, '35': 155.843, '36': 131.03, '37': 78.344, '38': 12.878, '39': -54.702, '40': -127.992, '41': -136.12}
[-136.12, -127.992, -54.702, 12.878, 78.344, 131.03, 155.843, 160.048, 433.742, 494.362, 582.801, 612.197, 624.688, 627.269, 884.448, 888.182, 915.571, 955.851, 1008.669, 1045.114, 1050.298, 1083.409, 1276.773, 1282.729, 1360.974, 1377.071, 1420.881, 1422.44, 1441.353, 1477.665, 1503.958, 1506.717, 1649.41

## Updating settings

In [10]:
broad_factor_rc=10.
settings_here = {'electrical': None, 'mechanical': None,
                 'Gamma_rc': broad_factor_rc, 'region': 1,
                 'font_dict': {'size': 18}, 'figsize': (12, 15), 'norm_max': 1e9, 'norm_min': 1e4,
                 'dynamic_range_n': 100}

## Acetic acid (B3LYP/cc-pVQZ)

In [ ]:
# selection of the spectrum
datain = data_vault.make_DatainputDict('gaussian', ('ACAC', 'B3LYP', 'cc_pVQZ'), wilson_root)
# list_figs = [(True, False), (False, True), (True, True)]
list_figs = [(True, True)]
region = 3
omega1 = np.arange(*regions[region][0])
omega2 = np.arange(*regions[region][1])
# omega1 = [1185.288, 1247.878, 1501.586]
# omega2 = [2687.219, 2491.481, 2364.564]

for s in list_figs:
    settings_here['electrical'] = s[0]
    settings_here['mechanical'] = s[1]
    vibEL=False
    directory='./'
    
    region = settings_here['region']
    Gamma_rc = settings_here['Gamma_rc']
    el_bool = settings_here['electrical']
    mech_bool = settings_here['mechanical']

    computedSpectrum = spectrum.SpectrumEVV(omega1, omega2, input_data_info=datain, vib_levels_harmonic=vibEL)
    computedSpectrum.addTerms(*terms_selection)
    print(computedSpectrum.fundamentals)
    print(sorted(list(computedSpectrum.fundamentals.values())))
    # print(computedSpectrum.tensor_3d.T)
    Gamma = spectrum.rec_cm2rec_s(Gamma_rc)
    # sec_hypol_data, savedict = computedSpectrum.intensity(Gamma, {}, el=el_bool, mech=mech_bool)

    sec_hypol_data = 0
    if settings_here['electrical']:
        el_gamma, Qab_contrib_dict = computedSpectrum.intensity_electrical(Gamma)
        sec_hypol_data += el_gamma

    if settings_here['mechanical']:
        mechall, Qabc_contrib_dict = computedSpectrum.intensity_mechanical(Gamma)
        template_array = np.full(mechall.shape, -0.+0.j, dtype=complex)
        sec_hypol_data += mechall

    name = rendering.make_name(datain, vibEL, settings_here, other, directory)
    artist = rendering.SpectrumFigure(sec_hypol_data, computedSpectrum.w1_mesh, computedSpectrum.w2_mesh, settings_here)
    title_on_top, text_under_the_figure = rendering.make_texts4fig(datain, computedSpectrum, artist, settings_here, other, directory)
    fig = artist.plot2Dmatplotlib(nametuple=(name, os.path.join(os.path.dirname('__file__')), title_on_top),
                                    text_under_the_figure=text_under_the_figure, diagonal=False, to_save=False)
    plt.show()

   code method basis_set                                   g16_3quanta_full
3  ACAC  B3LYP   cc_pVQZ  /home/vlew/Wilson/tests/test_database/dftGauss...

Used vibrational energy levels:
 harmonic? - False
{'0': 3560.764, '1': 3014.192, '2': 2955.502, '3': 1786.815, '4': 1436.403, '5': 1372.251, '6': 1324.394, '7': 1164.166, '8': 975.352, '9': 840.892, '10': 578.825, '11': 424.861, '12': 2966.938, '13': 1438.853, '14': 1049.419, '15': 645.672, '16': 533.342, '17': 51.769}
[51.769, 424.861, 533.342, 578.825, 645.672, 840.892, 975.352, 1049.419, 1164.166, 1324.394, 1372.251, 1436.403, 1438.853, 1786.815, 2955.502, 2966.938, 3014.192, 3560.764]
0/5832 -- 0.0
100/5832 -- 1.7146776406035664
200/5832 -- 3.429355281207133
300/5832 -- 5.1440329218107
Execution time -| electrical: 8.294668197631836 seconds
Electrical anharmonicities are calculated
0/5832 -- 0.0
1000/5832 -- 17.146776406035666
2000/5832 -- 34.29355281207133
3000/5832 -- 51.440329218106996



## Unique ω_1, ω_2 pairs - at resonances
lots of debug printing here

In [5]:
o1, o2 = (500., 3150., 10.), (500., 6050., 10.)

terms_selection = [0, 1], [0,]

omega1 = [1185.288, 1247.878, 1501.586]
omega2 = [2687.219, 2491.481, 2364.564]

datain = data_vault.make_DatainputDict('gaussian', ('ACDM', 'B3LYP', 'cc_pVQZ'), wilson_root)
vib_levels_harmonic = False
rec_cm=True
computedSpectrum = spectrum.SpectrumEVV(omega1, omega2, input_data_info=datain,
                                        vib_levels_harmonic=vib_levels_harmonic)
computedSpectrum.addTerms(*terms_selection)
factors = [1., 1., 0.5, 0.5, -0.5, -0.5]

dfs4terms_el, dfs4terms_mech = analysis.get_resonances_DF(computedSpectrum, rec_cm=rec_cm,
                                                 vib_levels_harmonic=vib_levels_harmonic)
pd.options.mode.chained_assignment = None
# pd.set_option('display.float_format', '{:.2g}'.format)
# pd.options.display.float_format = '{:.2e}'.format

frames = []

for dfMech in dfs4terms_mech:
    # how many resonances there could be - depends on the number of combinations of a,b,c - combinations_number = Nmodes**3
    # print the expressions for resonances of this term
    print('Resonances:', dfMech['res1'].iloc[0], '\nFormula:', dfMech['res2'].iloc[0], '\n')
    
    if rec_cm:
        # filters: non-zero F_abc; within the window selected above
        dfMech = dfMech[(dfMech['F_abc'] != 0.) & (dfMech['ω_2']>dfMech['ω_1'])
                        # & (abs(dfMech['ω_2']) > omega2[0]) & 
                        # (abs(dfMech['ω_2']) < omega2[-1]) & 
                        # (abs(dfMech['ω_1']) > omega1[0]) & 
                        # (abs(dfMech['ω_1']) < omega1[-1])
        ]
    else:
        dfMech = dfMech[(dfMech['F_abc'] != 0.) & (dfMech['ω_2']>dfMech['ω_1'])
                     # & (abs(dfMech['ω_2']) > spectrum.rec_cm2rec_s(omega2[0])) & 
                    # (abs(dfMech['ω_2']) < spectrum.rec_cm2rec_s(omega2[-1])) & 
                    # (abs(dfMech['ω_1']) > spectrum.rec_cm2rec_s(omega1[0])) & 
                    # (abs(dfMech['ω_1']) < spectrum.rec_cm2rec_s(omega1[-1]))
        ]
    
    Gamma = spectrum.rec_cm2rec_s(settings_here['Gamma_rc'])

    # adding a column for the sum term in product
    if rec_cm:
        dfMech['SoF'] = spectrum.rec_cm2rec_s(dfMech['FR1']+dfMech['FR2'])/spectrum.rec_cm2rec_s(dfMech['FR1'])/spectrum.rec_cm2rec_s(dfMech['FR2'])
        dfMech['Fermi'] = 1./spectrum.rec_cm2rec_s(dfMech['FR1'])/spectrum.rec_cm2rec_s(dfMech['FR2'])
        dfMech['abs Fermi'] = abs(1./spectrum.rec_cm2rec_s(dfMech['FR1'])/spectrum.rec_cm2rec_s(dfMech['FR2']))
        dfMech['DoR'] = np.real(dfMech['SoF']*(1./(- 1j * Gamma)/(- 1j * Gamma)))
    else:
        dfMech['SoF'] = (dfMech['FR1']+dfMech['FR2'])/dfMech['FR1']/dfMech['FR2']
        dfMech['Fermi'] = 1./dfMech['FR1']/dfMech['FR2']
        dfMech['abs Fermi'] = abs( 1./dfMech['FR1']/dfMech['FR2'])
        dfMech['DoR'] = np.real(dfMech['SoF']*(1./(- 1j * Gamma)/(- 1j * Gamma)))
    dfMech['pr'] = 1./(- 1j * Gamma)/(- 1j * Gamma)
    # dfMech['resonances'] is confirmed now, so the rest should be okay too; now confirmed!
    dfMech['gamma_mn'] = dfMech['avrg_g']*dfMech['F_abc']*dfMech['DoR']/computedSpectrum.tensor_3d.T[dfMech['a'], dfMech['b'], dfMech['c']]*(-1.)/48.
    dd = []
    for idx, row in dfMech.iterrows():
        d = {abs(row['avrg_g']): 'avrg_g', abs(row['F_abc']): 'F_abc', abs(row['DoR'])/abs(row['SoF']): 'res', abs(row['SoF']): 'SoF', computedSpectrum.tensor_3d.T[row['a'], row['b'], row['c']]: 'pref'}
        dd.append(d[max([abs(row['avrg_g']), abs(row['F_abc']), abs(row['DoR'])/abs(row['SoF']), abs(row['SoF']), computedSpectrum.tensor_3d.T[row['a'], row['b'], row['c']]])])
    dfMech['distribution'] = dd
    frames.append(dfMech)
    
result = pd.concat(frames)
result['abs'] = abs(result['gamma_mn'])**2
# print(result[['ii', 'a', 'b', 'c', 'ω_1', 'ω_2', 'FR1', 'FR2', 'SoF', 'abs Fermi', 'DoR', 'avrg_g', 'F_abc', 'gamma_mn', 'abs']].nlargest(7, 'abs Fermi'))

df = result[['ii', 'a', 'b', 'c', 'ω_1', 'ω_2', 'FR1', 'FR2', 'SoF', 'distribution', 'DoR', 'avrg_g', 'F_abc', 'gamma_mn', 'abs']]
formatted_df = df.copy()
# formatted_df['gamma_mn'] = df['gamma_mn'].map('{:.1e}'.format)
formatted_df['SoF'] = df['SoF'].map('{:.1f}'.format)
# formatted_df['ω_1'] = df['ω_1'].map('{:.1f}'.format)
# formatted_df['ω_2'] = df['ω_2'].map('{:.1f}'.format)
formatted_df['FR1'] = df['FR1'].map('{:.1f}'.format)
formatted_df['FR2'] = df['FR2'].map('{:.1f}'.format)
# formatted_df['abs Fermi'] = df['abs Fermi'].map('{:.2f}'.format)
# formatted_df['abs'] = df['abs'].map('{:.1e}'.format)
formatted_df['DoR'] = df['DoR'].map('{:.1e}'.format)
formatted_df['avrg_g'] = df['avrg_g'].map('{:.1e}'.format)
formatted_df['F_abc'] = df['F_abc'].map('{:.1e}'.format)

print(formatted_df.nlargest(7, 'abs'))


Used vibrational energy levels:
 harmonic? - False
Resonances: a+b,a__zero,a 
Formula: a+b+c,zero__c,a+b 

       ii   a   b   c       ω_1       ω_2     FR1     FR2      SoF distribution       DoR    avrg_g     F_abc      gamma_mn           abs
65094   0  36  37  36   223.427   422.617   644.0  -199.2   -761.1          res   3.7e+11  -2.0e-08  -8.1e-06 -1.368841e+06  1.873726e+12
70512   0  39  40  36   186.123   336.248   558.1  -112.8  -1552.1          res   7.5e+11   8.0e-09   3.2e-06 -7.397442e+05  5.472215e+11
65223   0  36  40  39   223.427   372.706   558.1  -186.6   -783.0          res   3.8e+11   8.0e-09   3.2e-06 -3.732064e+05  1.392830e+11
64656   0  36  27  18   223.427  1153.757  2562.1   258.6    934.4          res  -4.5e+11   1.5e-06  -1.7e-07 -8.356350e+04  6.982858e+09
15248   0   8  27   2  1774.116  2701.221  5411.4     5.4  40714.1          res  -2.0e+13   1.2e-05   6.8e-09  6.530024e+04  4.264121e+09
64310   0  36  19   8   223.427  1608.359  3383.8   165.8   1388

In [6]:
# smaller dataframe
result_later = (
    formatted_df
    .groupby(['ω_1', 'ω_2'])
    .agg({
        'gamma_mn': 'sum',
        'a': lambda x: list(x),
        'b': lambda x: list(x),
        'c': lambda x: list(x)
    })
    .reset_index()
)
result_later['final'] = abs(result_later['gamma_mn'])**2

result_later['abc_tuples'] = result_later.apply(lambda row: str(list(zip(row['a'], row['b'], row['c']))), axis=1)
result_later = result_later.drop(columns=['a', 'b', 'c'])
# print(result_later.sort_values(by=['final']))

           ω_1       ω_2      gamma_mn         final                                                                                                                                                                                                                                                                                                                                                            abc_tuples
821   1131.979  2547.242  3.086902e-14  9.528965e-28                                                                                                                                                                                                                                                       [(23, 18, 5), (23, 18, 15), (23, 18, 17), (23, 18, 22), (23, 18, 29), (23, 18, 34), (23, 18, 39), (23, 18, 40)]
947   1385.906  2789.153  3.614133e-12  1.306196e-23                                                                                                                              

## Plotting only resonances

In [33]:
import plotly.express as px
formatted_df1 = result_later.copy()
print(formatted_df1.columns)
# formatted_df['final'] = np.log10(result_later['final']) if result_later[result_later['final']]>1e4 else 0.
# formatted_df1['final'] = formatted_df1['final'].apply(lambda x: 0 if x < 1e4 else np.log(x))

fig = px.scatter(formatted_df1[(formatted_df1['final']>1e5)
                              # &(formatted_df['final']<1e8)
                 ], 
                 x='ω_1', 
                 y='ω_2', 
                 color='final', 
                 color_continuous_scale='viridis_r', 
                 title='Scatter Plot of final vs ω_1 and ω_2',
                 width=1300, height=800,
                 hover_data={'final':':.2e', 'abc_tuples': True, #'a': True, 'b': True, 'c': True
                             },
                 range_color=(5e4,3e12))

fig.update_layout(
    xaxis_title='ω_1',
    yaxis_title='ω_2',
    coloraxis_colorbar=dict(title='final'),
    plot_bgcolor='aliceblue'
)

fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightPink')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightPink')
fig.update_coloraxes(colorbar_tickformat = '.2e'.format())

fig.show()

Index(['ω_1', 'ω_2', 'gamma_mn', 'final', 'abc_tuples'], dtype='object')
